# Lens B — Geographic Power

Who researches whom? This notebook measures "parachute science" — when researchers from high-income countries study populations in low- and middle-income countries without local authorship — and maps the flow of knowledge production.

### Analyses
1. **Parachute science index by year** — proportion of papers with no local first or last author
2. **First-author country × study-country flow matrix** — for arc map visualization
3. **Regional trajectory** — parachute index by WHO region 2010–2024
4. **Subtopic-level parachute analysis** — does it vary by research type?
5. **Funder nationality × parachute index correlation**

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

DB = 'data/global_health.duckdb'
con = duckdb.connect(DB, read_only=True)

## Load core data

For each paper we need:
- The study country (where the research was conducted)
- The first author's institutional country
- The last author's institutional country

In [ ]:
# Topic taxonomy for labels
topic_labels = pd.read_csv('data/taxonomy/topic_taxonomy.csv')
cat_names = topic_labels.drop_duplicates('category_letter')[['category_letter', 'category_name']]
cat_map = dict(zip(cat_names['category_letter'], cat_names['category_name']))

# WHO region mapping (ISO-2 country code -> WHO region)
# This is a simplified mapping for the most common countries in the dataset
WHO_REGIONS = {
    # African Region (AFRO)
    'ZA': 'AFRO', 'NG': 'AFRO', 'KE': 'AFRO', 'ET': 'AFRO', 'TZ': 'AFRO',
    'GH': 'AFRO', 'UG': 'AFRO', 'MW': 'AFRO', 'ZM': 'AFRO', 'RW': 'AFRO',
    'MZ': 'AFRO', 'SN': 'AFRO', 'CM': 'AFRO', 'CD': 'AFRO', 'BF': 'AFRO',
    'ML': 'AFRO', 'ZW': 'AFRO', 'NE': 'AFRO', 'CI': 'AFRO', 'BJ': 'AFRO',
    'MG': 'AFRO', 'SL': 'AFRO', 'LR': 'AFRO', 'GM': 'AFRO', 'TD': 'AFRO',
    'AO': 'AFRO', 'SS': 'AFRO', 'BI': 'AFRO', 'SO': 'AFRO', 'ER': 'AFRO',
    'NA': 'AFRO', 'BW': 'AFRO', 'LS': 'AFRO', 'SZ': 'AFRO', 'TG': 'AFRO',
    # Americas (AMRO/PAHO)
    'US': 'AMRO', 'CA': 'AMRO', 'BR': 'AMRO', 'MX': 'AMRO', 'CO': 'AMRO',
    'AR': 'AMRO', 'PE': 'AMRO', 'CL': 'AMRO', 'GT': 'AMRO', 'HN': 'AMRO',
    'HT': 'AMRO', 'DO': 'AMRO', 'JM': 'AMRO', 'BO': 'AMRO', 'EC': 'AMRO',
    # South-East Asia (SEARO)
    'IN': 'SEARO', 'BD': 'SEARO', 'NP': 'SEARO', 'LK': 'SEARO', 'MM': 'SEARO',
    'TH': 'SEARO', 'ID': 'SEARO', 'TL': 'SEARO',
    # European Region (EURO)
    'GB': 'EURO', 'DE': 'EURO', 'FR': 'EURO', 'IT': 'EURO', 'ES': 'EURO',
    'NL': 'EURO', 'SE': 'EURO', 'NO': 'EURO', 'DK': 'EURO', 'FI': 'EURO',
    'CH': 'EURO', 'BE': 'EURO', 'AT': 'EURO', 'IE': 'EURO', 'PT': 'EURO',
    'PL': 'EURO', 'GR': 'EURO', 'CZ': 'EURO', 'HU': 'EURO', 'RO': 'EURO',
    'TR': 'EURO', 'IL': 'EURO', 'RU': 'EURO',
    # Eastern Mediterranean (EMRO)
    'EG': 'EMRO', 'PK': 'EMRO', 'AF': 'EMRO', 'IR': 'EMRO', 'IQ': 'EMRO',
    'SA': 'EMRO', 'JO': 'EMRO', 'LB': 'EMRO', 'SD': 'EMRO', 'YE': 'EMRO',
    'SY': 'EMRO', 'LY': 'EMRO', 'TN': 'EMRO', 'MA': 'EMRO', 'OM': 'EMRO',
    'AE': 'EMRO', 'QA': 'EMRO', 'BH': 'EMRO', 'KW': 'EMRO', 'PS': 'EMRO',
    'SO': 'EMRO', 'DJ': 'EMRO',
    # Western Pacific (WPRO)
    'CN': 'WPRO', 'JP': 'WPRO', 'AU': 'WPRO', 'NZ': 'WPRO', 'KR': 'WPRO',
    'PH': 'WPRO', 'VN': 'WPRO', 'MY': 'WPRO', 'KH': 'WPRO', 'LA': 'WPRO',
    'PG': 'WPRO', 'FJ': 'WPRO', 'SG': 'WPRO', 'MN': 'WPRO',
}

# Core dataset: works with study country + first/last author countries
parachute_data = con.execute("""
    SELECT
        w.openalex_id,
        w.publication_year,
        w.topic_category,
        w.topic_subtopic,
        w.study_country,
        first_auth.institution_country AS first_author_country,
        last_auth.institution_country AS last_author_country
    FROM works w
    LEFT JOIN (
        SELECT openalex_id, institution_country
        FROM authorships WHERE position = 'first'
    ) first_auth ON w.openalex_id = first_auth.openalex_id
    LEFT JOIN (
        SELECT openalex_id, institution_country
        FROM authorships WHERE position = 'last'
    ) last_auth ON w.openalex_id = last_auth.openalex_id
    WHERE w.study_country IS NOT NULL
      AND w.topic_category IS NOT NULL
""").fetchdf()

# Compute parachute flag: paper has a study country but neither first
# nor last author is from that country
parachute_data['is_parachute'] = (
    (parachute_data['first_author_country'] != parachute_data['study_country']) &
    (parachute_data['last_author_country'] != parachute_data['study_country'])
)

# Handle NaN countries: if both author countries are missing, mark as unknown
both_missing = (
    parachute_data['first_author_country'].isna() &
    parachute_data['last_author_country'].isna()
)
parachute_data.loc[both_missing, 'is_parachute'] = np.nan

print(f'Papers with study country: {len(parachute_data):,}')
valid = parachute_data.dropna(subset=['is_parachute'])
print(f'Papers with author + study country data: {len(valid):,}')
print(f'Parachute papers: {valid["is_parachute"].sum():,.0f} ({valid["is_parachute"].mean():.1%})')

---
## B1. Parachute Science Index by Year

In [ ]:
valid = parachute_data.dropna(subset=['is_parachute'])

parachute_by_year = (
    valid
    .groupby('publication_year')
    .agg(
        total=('openalex_id', 'count'),
        parachute=('is_parachute', 'sum'),
    )
    .assign(parachute_index=lambda x: x['parachute'] / x['total'])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(parachute_by_year['publication_year'], parachute_by_year['parachute_index'],
        'o-', color='#d62728', linewidth=2, markersize=8)
ax.fill_between(parachute_by_year['publication_year'], parachute_by_year['parachute_index'],
                alpha=0.15, color='#d62728')
ax.set_xlabel('Year')
ax.set_ylabel('Parachute Science Index')
ax.set_title('Parachute Science Index Over Time\n(Proportion of papers with no local first or last author)')
ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()

parachute_by_year

---
## B2. First-Author Country × Study-Country Flow Matrix

Who studies whom? This flow matrix reveals the dominant researcher–subject relationships.

In [ ]:
# Flow matrix: first-author country -> study country
flow = (
    valid
    .groupby(['first_author_country', 'study_country'])
    ['openalex_id'].count()
    .reset_index(name='n_papers')
    .sort_values('n_papers', ascending=False)
)

# Top 15 author countries and top 15 study countries for the heatmap
top_author_countries = flow.groupby('first_author_country')['n_papers'].sum().nlargest(15).index
top_study_countries = flow.groupby('study_country')['n_papers'].sum().nlargest(15).index

flow_matrix = flow[
    flow['first_author_country'].isin(top_author_countries) &
    flow['study_country'].isin(top_study_countries)
].pivot_table(
    index='first_author_country', columns='study_country',
    values='n_papers', fill_value=0
)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(flow_matrix, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white')
ax.set_xlabel('Study Country')
ax.set_ylabel('First Author Country')
ax.set_title('Research Flow: First-Author Country → Study Country')
fig.tight_layout()
plt.show()

# Top 20 cross-border flows
cross_border = flow[flow['first_author_country'] != flow['study_country']].head(20)
print('\nTop 20 cross-border research flows:')
cross_border

---
## B3. Regional Trajectory — Parachute Index by WHO Region

In [ ]:
valid_with_region = valid.copy()
valid_with_region['study_region'] = valid_with_region['study_country'].map(WHO_REGIONS)
valid_with_region = valid_with_region.dropna(subset=['study_region'])

regional_parachute = (
    valid_with_region
    .groupby(['publication_year', 'study_region'])
    .agg(
        total=('openalex_id', 'count'),
        parachute=('is_parachute', 'sum'),
    )
    .assign(parachute_index=lambda x: x['parachute'] / x['total'])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
for region in sorted(regional_parachute['study_region'].unique()):
    rd = regional_parachute[regional_parachute['study_region'] == region]
    ax.plot(rd['publication_year'], rd['parachute_index'], 'o-', label=region, markersize=5)

ax.set_xlabel('Year')
ax.set_ylabel('Parachute Science Index')
ax.set_title('Parachute Science Index by WHO Region Over Time')
ax.legend(title='Study Region')
ax.set_ylim(0, 1)
fig.tight_layout()
plt.show()

---
## B4. Subtopic-Level Parachute Analysis

Does parachute science vary by research topic? Some fields may be more locally driven than others.

In [ ]:
valid['topic_name'] = valid['topic_category'].map(cat_map)

parachute_by_topic = (
    valid
    .groupby('topic_name')
    .agg(
        total=('openalex_id', 'count'),
        parachute=('is_parachute', 'sum'),
    )
    .assign(parachute_index=lambda x: x['parachute'] / x['total'])
    .sort_values('parachute_index', ascending=True)
)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#d62728' if x > parachute_by_topic['parachute_index'].median() else '#2ca02c'
          for x in parachute_by_topic['parachute_index']]
parachute_by_topic['parachute_index'].plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Parachute Science Index')
ax.set_title('Parachute Science Index by Topic Category')
ax.axvline(x=parachute_by_topic['parachute_index'].median(), color='gray',
           linestyle='--', alpha=0.7, label='Median')
ax.legend()
for i, (idx, row) in enumerate(parachute_by_topic.iterrows()):
    ax.text(row['parachute_index'] + 0.01, i, f'{row["parachute_index"]:.0%}',
            va='center', fontsize=9)
fig.tight_layout()
plt.show()

---
## B5. Funder Nationality × Parachute Index Correlation

Do papers funded by foreign funders have higher parachute rates than locally funded papers?

In [ ]:
# Join funder info to parachute data
funded_parachute = con.execute("""
    SELECT
        w.openalex_id,
        w.study_country,
        w.topic_category,
        f.funder_country,
        first_auth.institution_country AS first_author_country,
        last_auth.institution_country AS last_author_country
    FROM works w
    JOIN grants g ON w.openalex_id = g.openalex_id
    JOIN funders f ON REPLACE(g.funder_id, 'https://openalex.org/', '') = f.openalex_id
    LEFT JOIN (
        SELECT openalex_id, institution_country
        FROM authorships WHERE position = 'first'
    ) first_auth ON w.openalex_id = first_auth.openalex_id
    LEFT JOIN (
        SELECT openalex_id, institution_country
        FROM authorships WHERE position = 'last'
    ) last_auth ON w.openalex_id = last_auth.openalex_id
    WHERE w.study_country IS NOT NULL
""").fetchdf()

funded_parachute['is_parachute'] = (
    (funded_parachute['first_author_country'] != funded_parachute['study_country']) &
    (funded_parachute['last_author_country'] != funded_parachute['study_country'])
)

# Is the funder from the same country as the study?
funded_parachute['funder_is_local'] = (
    funded_parachute['funder_country'] == funded_parachute['study_country']
)

funder_parachute_rates = (
    funded_parachute
    .groupby('funder_is_local')
    .agg(
        total=('openalex_id', 'count'),
        parachute=('is_parachute', 'sum'),
    )
    .assign(parachute_index=lambda x: x['parachute'] / x['total'])
)
funder_parachute_rates.index = ['Foreign funder', 'Local funder']

fig, ax = plt.subplots(figsize=(8, 4))
funder_parachute_rates['parachute_index'].plot(kind='barh', ax=ax, color=['#d62728', '#2ca02c'])
ax.set_xlabel('Parachute Science Index')
ax.set_title('Parachute Science Rate by Funder Locality')
for i, (idx, row) in enumerate(funder_parachute_rates.iterrows()):
    ax.text(row['parachute_index'] + 0.01, i,
            f'{row["parachute_index"]:.0%} (n={row["total"]:,})', va='center')
fig.tight_layout()
plt.show()

---
## Save Results to DuckDB

In [ ]:
con.close()

con_write = duckdb.connect(DB)

# Parachute index by year
con_write.execute('CREATE OR REPLACE TABLE parachute_index AS SELECT * FROM ?', [parachute_by_year])

# Author-study country flow matrix
con_write.execute('CREATE OR REPLACE TABLE author_study_country_flows AS SELECT * FROM ?', [flow])

# Regional parachute trends
con_write.execute('CREATE OR REPLACE TABLE parachute_by_region AS SELECT * FROM ?', [regional_parachute])

con_write.close()
print('Lens B results saved to DuckDB.')